# 📊 10｜台北租屋：先抓異常值，再用每坪租金公平比較

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/roberthsu2003/PythonForDataAnalysis/blob/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/10_rent_seaborn/lesson.ipynb)

> 💡 **學習模式說明**：
> 本筆記本專為**完全不寫程式的初學者**設計，全程搭配 **Google Colab 內建的 Gemini AI** 進行協作。
> 你不需要手寫程式碼：先用 Prompt 指揮 AI，再對照下方已附的參考程式，執行並驗收數字與圖表。
> 🆕 **本例只新增一個重點**：少數**輸入錯誤**（月租 1 元、98 萬元、0 坪）會毀掉整張分布圖；先找出並排除、留下排除清單，再衍生**每坪租金**，讓大小不同的物件可以公平比較。

## 🪜 本例路線：先完成一次，再逐步加深
1. **必做主線（單元一～三）**：用描述統計找出 3 筆異常、排除並留下清單，衍生每坪租金，最後用**一張直方圖**回答「台北每坪租金大多落在哪裡？」。單元三後先停下來驗收。
2. **報表應用（單元四）**：匯出清理後的物件資料與排除清單。
3. **選做挑戰（單元五～六）**：比較各行政區的每坪租金，或觀察坪數與月租金的關係。

---

## 🧭 資料專案介紹：貫徹「看懂資料與選對圖表」思維 SOP

資料：`data/rent_listings.csv`（請先閱讀[本例資料卡](./data/DATA_CARD.md)）。**一列代表：一筆出租物件**，共 800 筆，涵蓋台北市 6 個行政區。

### 1. 欄位角色貼標籤
- ⚪ **標識 (ID)**：`物件編號` ➔ 只能計數或辨識。
- 🔵 **條件 (Dimension)**：`行政區`、`類型`、`電梯`（類別）；`樓層`（數字，但當作條件）。
- 🟢 **指標 (Metric)**：`坪數`（坪）、`屋齡`（年）、`月租金`（元）；衍生指標 `每坪租金`（元／坪）。

### 2. 五大提問積木與核心問題
- 📊 **分布（必做）**：台北出租物件的每坪租金大多落在哪個區間？
- 🏙️ **比較＋分布（選做）**：哪個行政區的每坪租金最高？各區的範圍差多少？
- 🔗 **關係（選做）**：坪數越大，月租金越高嗎？不同類型的物件有不同的模式嗎？

### 3. 邊界反向檢核防呆（不可觸犯的紅線）
- ⚠️ **先看最小值與最大值**：月租金 1 元、980,000 元與 0 坪是輸入錯誤；不排除的話，直方圖會被壓扁、平均被拉高。
- ⚠️ **排除要留紀錄**：不得刪改原始檔；清理後另存新檔，並附上「排除清單」與理由，讓別人可以檢查。
- ⚠️ **異常 ≠ 錯誤**：真正的豪宅高租金不是錯誤；只排除「不可能」的值，不為了圖好看而刪除高價物件。
- ⚠️ **比較要公平**：整層住家比套房大很多，直接比月租金不公平；比較行政區時用每坪租金。
- ⚠️ **刊登租金 ≠ 成交租金**：這是刊登資料，不代表實際成交價；也不能推論「住大安區一定比較貴」適用於每一間房。

### 4. 決定最適圖表
- **每坪租金集中在哪裡** ➔ **直方圖**（連續數值看分布；加上中位數線，標明有效物件數）。
- **各行政區每坪租金（選做）** ➔ **箱形圖**（多組比較分布，依中位數排序）。
- **坪數 vs 月租金（選做）** ➔ **散佈圖**（依類型上色，點不連線）。

---

## 🤖 Colab Gemini AI 操作方式
1. 每次只複製一個單元的 Prompt，貼到 Colab Gemini 對話框。
2. 比較 AI 的答覆與下方參考程式，執行目前單元的儲存格。
3. 先看懂輸出並回答驗收問題，再進入下一單元。

In [ ]:
# 第 0 步：下載範例資料並安裝繁體中文字型（請先點擊左側 ▶ 執行此儲存格）
import os
from pathlib import Path
from urllib.request import urlretrieve

# 1. 下載資料（已存在就不重複下載；若下載失敗，請手動上傳到 data/）
data_dir = Path('data')
data_dir.mkdir(exist_ok=True)
files = {
    'rent_listings.csv': 'https://raw.githubusercontent.com/roberthsu2003/PythonForDataAnalysis/master/AI%E5%8D%94%E4%BD%9C%E8%B3%87%E6%96%99%E5%88%86%E6%9E%90_12%E5%80%8B%E7%AF%84%E4%BE%8B/10_rent_seaborn/data/rent_listings.csv',
}
for name, url in files.items():
    if not (data_dir / name).exists():
        urlretrieve(url, data_dir / name)
print('📂 data/ 內的檔案：', sorted(p.name for p in data_dir.iterdir()))

# 2. 安裝與設定 Colab 繁體中文字型 (思源黑體)
%pip install -q mplfonts
import glob
import mplfonts
from matplotlib import font_manager
from mplfonts import use_font

for f in glob.glob(os.path.join(os.path.dirname(mplfonts.__file__), "fonts", "*.[ot]tf")):
    font_manager.fontManager.addfont(f)
use_font("Noto Sans CJK SC")

print("✅ 環境初始化完成！資料已就緒，繁體中文字型已設定成功。請開始進行下方單元。")

### 📝 單元一【必做】：讀取資料，用描述統計找出可疑數值（貼標籤）

> 🎯 **思維重點**：畫圖前先看「最小值、最大值」：數字格式正確，不代表數值合理。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請幫我用 pandas 讀取 data/rent_listings.csv：
1. 顯示前 5 筆資料、所有欄位名稱、總列數與各欄缺值數。
2. 對坪數、屋齡、樓層、月租金計算描述統計（筆數、平均、最小、四分位數、最大）。
3. 依類型列出筆數與月租金中位數。
4. 用文字回報：哪些欄位是標識、條件、指標？描述統計中有哪些數字明顯不合理？
先不要修改資料，也不要畫圖。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
import pandas as pd

df = pd.read_csv('data/rent_listings.csv')
print("--- 前 5 筆資料 ---")
display(df.head())
print(f"\n總列數：{len(df)}；缺值：{df.isna().sum().sum()}")

print("\n--- 描述統計（注意最小值與最大值）---")
display(df[['坪數', '屋齡', '樓層', '月租金']].describe().round(1))
print("各類型筆數與月租金中位數：")
display(df.groupby('類型')['月租金'].agg(['count', 'median']))

### 📝 單元二【必做】：邊界檢核：排除不可能的值、留下清單，衍生每坪租金

> 🎯 **思維重點**：只排除「不可能」的值，並留下排除清單；再衍生「每坪租金」。比較排除前後的平均，親眼看見 3 筆錯誤的影響。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請依下列規則處理異常值，並說明理由：
1. 定義不合理的資料：坪數 ≤ 0、月租金 < 1,000 元、月租金 > 500,000 元。在註解說明為什麼這些是「輸入錯誤」而不是「真實的高價或低價」。
2. 列出符合條件的物件（排除清單），並註明各自的排除原因。
3. 建立排除後的新資料表（不要修改原始檔），印出排除前後筆數，以及排除前後的月租金平均與中位數。
4. 在新資料表中新增「每坪租金」＝月租金 ÷ 坪數，四捨五入到整數。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
# 不合理的判斷規則：坪數不可能 ≤ 0；台北月租不可能低於 1,000 元或超過 50 萬元
# （真正的豪宅高租金不會到 98 萬元一個月；1 元與 0 坪顯然是漏打或打錯）
reasons = pd.Series('', index=df.index)
reasons[df['坪數'] <= 0] += '坪數為 0；'
reasons[df['月租金'] < 1000] += '月租金低於 1,000 元；'
reasons[df['月租金'] > 500000] += '月租金超過 50 萬元；'

excluded = df[reasons != ''].assign(排除原因=reasons[reasons != ''])
print("--- 排除清單 ---")
display(excluded)

clean = df[reasons == ''].copy()
print(f"排除前 {len(df)} 筆 → 排除後 {len(clean)} 筆")
print(f"月租金平均：{df['月租金'].mean():,.0f} → {clean['月租金'].mean():,.0f} 元")
print(f"月租金中位數：{df['月租金'].median():,.0f} → {clean['月租金'].median():,.0f} 元（中位數幾乎不受影響）")

clean['每坪租金'] = (clean['月租金'] / clean['坪數']).round(0)
display(clean[['物件編號', '行政區', '類型', '坪數', '月租金', '每坪租金']].head())

### 📝 單元三【必做】：提問積木【分布】— 每坪租金大多落在哪裡？（直方圖）

> 🎯 **思維重點**：連續數值看集中區間，用【直方圖】；加上中位數線，標題寫出有效物件數 n。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請用 seaborn 畫出清理後「每坪租金」的直方圖：
1. X 軸為每坪租金（元／坪），Y 軸為物件數；區間寬度設為 100 元。
2. 畫出中位數的垂直參考線，並在圖例中標示數值。
3. 標題寫出有效物件數 n，圖表文字全部使用繁體中文。
4. 另外印出每坪租金的描述統計，用數字描述主要集中的區間（例如四分位數範圍）。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
import matplotlib.pyplot as plt
import seaborn as sns

median_ppp = clean['每坪租金'].median()
plt.figure(figsize=(9, 5))
sns.histplot(data=clean, x='每坪租金', binwidth=100, color='#3498db')
plt.axvline(median_ppp, color='#e74c3c', linestyle='--', linewidth=2, label=f'中位數 {median_ppp:,.0f} 元／坪')
plt.title(f'台北出租物件每坪租金分布（n = {len(clean)}）', fontsize=14, fontweight='bold')
plt.xlabel('每坪租金（元／坪，每 100 元一組）')
plt.ylabel('物件數')
plt.legend()
plt.show()

desc = clean['每坪租金'].describe()
display(desc.round(0))
print(f"💡 一半的物件落在 {desc['25%']:,.0f}～{desc['75%']:,.0f} 元／坪之間。")

## ✅ 必做主線驗收：現在先停下來
- 描述統計顯示月租金最小 1 元、最大 980,000 元，坪數最小 0 坪；這 3 筆列入排除清單，剩 **797 筆**。
- 排除前月租金平均 19,983 元，排除後 18,818 元——3 筆錯誤就讓平均差了 1,165 元；中位數幾乎不變（13,850 → 13,900）。
- 每坪租金中位數 **1,249 元／坪**，一半的物件落在 **1,055～1,462 元／坪**之間。
- 用自己的話回答：如果有一筆月租 20 萬元的豪宅，它算輸入錯誤嗎？要怎麼判斷？

若今天繼續做報表，請進行**單元四**；單元五、六是選做挑戰。

---

### 📝 單元四【報表應用】：匯出清理後資料與排除清單

> 🎯 **思維重點**：清理結果要能被檢查：乾淨資料與排除清單放在同一個檔案，任何人都能回頭核對。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請將清理結果匯出成 Excel：
1. 存為 output/租屋清理結果.xlsx（若 output 目錄不存在請自動建立）。
2. 工作表「清理後物件」：清理後資料（含每坪租金）；工作表「排除清單」：被排除的物件與排除原因。
3. 匯出後重新讀取，核對兩個工作表的筆數相加等於原始 800 筆。
4. 提供 Colab 下載該 Excel 檔案的程式碼。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
os.makedirs('output', exist_ok=True)
output_path = 'output/租屋清理結果.xlsx'
with pd.ExcelWriter(output_path) as writer:
    clean.to_excel(writer, sheet_name='清理後物件', index=False)
    excluded.to_excel(writer, sheet_name='排除清單', index=False)

kept = pd.read_excel(output_path, sheet_name='清理後物件')
removed = pd.read_excel(output_path, sheet_name='排除清單')
print(f"清理後 {len(kept)} 筆 + 排除 {len(removed)} 筆 = {len(kept) + len(removed)} 筆（原始 {len(df)} 筆）")

try:
    from google.colab import files
    files.download(output_path)
    print("✅ 已啟動瀏覽器下載 Excel 檔案！")
except Exception:
    print(f"✅ Excel 檔案已儲存於：{output_path}")

### 📝 單元五【選做挑戰】：提問積木【比較＋分布】— 各行政區的每坪租金（箱形圖）

> 🎯 **思維重點**：比較 6 個行政區，看的不只是平均，而是整個分布；用【箱形圖】並依中位數排序，比較的指標是公平的「每坪租金」。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請用 seaborn 比較各行政區的每坪租金：
1. 依行政區計算物件數、每坪租金中位數，依中位數由高到低排序。
2. 畫出箱形圖：X 軸為行政區（依中位數排序），Y 軸為每坪租金（元／坪）。
3. 用文字指出中位數最高與最低的行政區，並提醒：這是刊登租金，個別物件仍有很大差異。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
order = clean.groupby('行政區')['每坪租金'].median().sort_values(ascending=False).index
area_stats = clean.groupby('行政區')['每坪租金'].agg(物件數='count', 中位數='median').loc[order]
display(area_stats)

plt.figure(figsize=(9, 5))
sns.boxplot(data=clean, x='行政區', y='每坪租金', order=order, color='#a8dadc')
plt.title('各行政區每坪租金分布（依中位數排序）', fontsize=13, fontweight='bold')
plt.xlabel('行政區')
plt.ylabel('每坪租金（元／坪）')
plt.show()
print(f"💡 中位數最高：{order[0]}（{area_stats['中位數'].iloc[0]:,.0f}），最低：{order[-1]}（{area_stats['中位數'].iloc[-1]:,.0f}）元／坪。")
print("⚠️ 邊界提醒：這是刊登租金；同一區內差異也很大，不能說該區每一間都比較貴。")

### 📝 單元六【選做挑戰】：提問積木【關係】— 坪數越大，月租金越高嗎？（散佈圖）

> 🎯 **思維重點**：兩個連續數值看連動，用【散佈圖】；依類型上色，就能看出「整層住家」自成一群——混在一起看的相關係數，可能藏著不同類型的差異。

📋 **複製以下 Prompt 貼給 Colab Gemini**：
```text
請用 seaborn 分析清理後資料中「坪數」與「月租金」的關係：
1. 畫散佈圖：X 軸為坪數（坪）、Y 軸為月租金（元），依類型用不同顏色，點不要連線。
2. 計算全部物件，以及各類型分別的皮爾森相關係數。
3. 用文字解讀：坪數與月租金是否正相關？不同類型的分布位置有何不同？並附註「相關不等於因果」。
```

In [ ]:
# 👉 請將 Gemini 產生的程式碼貼於此處執行
overall = clean['坪數'].corr(clean['月租金'])
by_type = clean.groupby('類型')[['坪數', '月租金']].apply(lambda g: g['坪數'].corr(g['月租金'])).round(2)

plt.figure(figsize=(9, 5.5))
sns.scatterplot(data=clean, x='坪數', y='月租金', hue='類型', alpha=0.7)
plt.title(f'坪數與月租金（全部物件 r = {overall:.2f}）', fontsize=13, fontweight='bold')
plt.xlabel('坪數（坪）')
plt.ylabel('月租金（元）')
plt.show()
print("各類型的相關係數：", by_type.to_dict())
print("⚠️ 邊界提醒：坪數與租金相關，但租金還受行政區、屋齡等影響；相關不等於因果。")

---

## 🎯 學習驗收與成果回顧

完成目前選擇的層次後，請檢視你是否能親自驗收：
- [ ] **必做**：排除 3 筆輸入錯誤剩 797 筆；每坪租金直方圖標示中位數與 n。
- [ ] **報表應用**：Excel 同時有清理後物件與排除清單，筆數相加為 800。
- [ ] **選做挑戰**：行政區箱形圖依中位數排序；散佈圖依類型上色並說明相關不等於因果。
- [ ] **AI 協作**：能說明自己訂的排除規則、AI 幫忙執行的事，以及自己如何確認結果。